# MATH840 final project: final submission notebook

Run all cells top to bottom (about 8 to 12 minutes on a Colab GPU runtime; the CPU runtime also works but the Chronos-2 part is slower).
Upload `train.csv` and `weather_forecast.csv` when asked. It writes and downloads `Boichuk_project.csv`.

**Point forecast:** a blend of
- the mean of four gradient-boosting models (LightGBM raw, LightGBM log1p, LightGBM ratio with recency weights, XGBoost raw), trained **without** `cloud_cover` and **with** day-level weather features, and
- the mean of two **Chronos-2** variants that read the last 1024 / 2048 hours of output together with the weather (past and forecast; no cloud_cover).

The default weights are 30% GBM / 70% Chronos-2 (`WG`). After that, night hours are multiplied by `NF` = 0.85 (it lowers MAPE, which is dominated by near-zero night hours). No shrinking towards the seasonal naive.

**Re-running:** best is Runtime > Restart and run all. Every cell is safe to re-run on its own now (no cumulative scaling, no self-referencing function).

**Interval:** several recipes are compared on the recent backtest origins with a scale-free score (the Winkler of the band divided by the Winkler of the benchmark's own band,
on a log scale, because the leaderboard ranks per window), and the best one with day-2 coverage of at least 94% is used.
Recipes: residual quantiles (absolute, relative to the forecast level, or a mix), calibrated online, and Chronos-2's own quantiles.

**Why `cloud_cover` is dropped:** the brief says 0 = clear and 1 = overcast, but in `train.csv` humid hours have low cloud values and higher cloud goes with higher output (the opposite), while in `weather_forecast.csv`
Oct 27 has cloud 1.0 with humidity of about 100%. The two files do not agree, and a model that trusts the column forecasts day 1 too high.

**Validation:** 120 origins (the recent 60 midnights and 60 from last autumn), models refitted every 10 origins on data known at that time, forecast-weather error simulated (its size is an assumption).

In [ ]:
QUICK = False            # True = fast smoke test with small models
RUN_FOUNDATION = True    # Chronos-2 (needs `pip install chronos-forecasting`, downloads about 0.5 GB)
WG = 0.3                 # weight of the GBM ensemble in the point forecast (1 - WG goes to Chronos-2)
NF, DF = 0.85, 1.0       # multiply the night / day hours of the forecast (night 0.85 lowers MAPE; 1.0, 1.0 = off)
FORCE_RECIPE = ("mix", "4", 1.2)   # interval recipe to use; None = take the best one found in section 5
SAVE = True              # write and download the CSV at the end
SURNAME = "Boichuk"      # the file is {SURNAME}_project.csv; check the name the template asks for
NSEG = 20 if QUICK else 60
BLOCK = 10

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 150, "display.max_columns", 40, "display.precision", 3)

NEEDED = ["train.csv", "weather_forecast.csv"]
if not all(Path(f).exists() for f in NEEDED):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put {NEEDED} next to this notebook.")

try:
    import lightgbm as lgb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb

data = pd.read_csv("train.csv", parse_dates=["ds"])
wfc = pd.read_csv("weather_forecast.csv", parse_dates=["ds"])

H = 48
WX = ["cloud_cover", "temperature", "wind_speed", "pressure", "humidity", "daylight", "air_density"]
WEATHER6 = [c for c in WX if c != "daylight"]          # daylight is a calendar fact, the rest is weather
assert len(wfc) == H and wfc["ds"].iloc[0] == data["ds"].iloc[-1] + pd.Timedelta(hours=1)
print(f"history: {len(data)} hours, {data['ds'].min()} to {data['ds'].max()}")
print(f"forecast: {wfc['ds'].iloc[0]} to {wfc['ds'].iloc[-1]}")

In [ ]:
# The three scores and the benchmark, exactly as in the course template.
def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


def mape(actual, forecast) -> float:
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return float(np.mean(np.abs((actual - forecast) / actual)) * 100)


def winkler(actual, lower, upper, alpha: float = 0.05) -> float:
    actual, lower, upper = (np.asarray(a, float) for a in (actual, lower, upper))
    below = np.clip(lower - actual, 0, None)
    above = np.clip(actual - upper, 0, None)
    return float(np.mean((upper - lower) + 2 / alpha * (below + above)))


def snaive(history, h: int = H, m: int = 24):
    history = np.asarray(history, float)
    point = np.array([history[-m + (i % m)] for i in range(h)])
    sigma = np.std(history[m:] - history[:-m], ddof=1)
    k = np.arange(h) // m
    half = 1.96 * sigma * np.sqrt(k + 1)
    return point, np.clip(point - half, 0, None), point + half

In [ ]:
y = data["y"].to_numpy(float)
n = len(y)
ds = data["ds"]
hour = ds.dt.hour.to_numpy()
W = {c: data[c].to_numpy(float) for c in WX}
K = np.arange(1, H + 1)

s_ = pd.Series(y)
LEVEL = {w: s_.rolling(w).mean().to_numpy() for w in (24, 72, 168, 336)}
PEAK168 = s_.rolling(168).max().to_numpy()
night = 1.0 - W["daylight"]
NIGHT168 = (pd.Series(y * night).rolling(168).sum() / pd.Series(night).rolling(168).sum()).to_numpy()
L = LEVEL[168]                                              # the 'level' used to scale ratio models

# origins: every midnight with a full history behind it and 48 known hours after it
idx_all = np.arange(n)
ORIG_ALL = np.where((hour == 0) & (idx_all >= 720) & (idx_all + H <= n - 1))[0]
NVAL, BLOCK = 60, 10
VAL = ORIG_ALL[-NVAL:]
print("training origins available:", len(ORIG_ALL), "| validation origins:", len(VAL),
      "from", ds.iloc[VAL[0]].date(), "to", ds.iloc[VAL[-1]].date())


def hist_wx(orig):
    orig = np.asarray(orig)
    return {c: W[c][orig[:, None] + K[None, :]] for c in WX}


def perturb(wx, rng):
    # Simulated weather-forecast error, growing with the horizon (an assumption, see the text above).
    g = (K / H)[None, :]
    out = {c: v.copy() for c, v in wx.items()}
    shape = wx["cloud_cover"].shape
    out["cloud_cover"] = np.clip(np.round((wx["cloud_cover"] + rng.normal(0, 0.15 + 0.25 * g, shape)) * 4) / 4, 0, 1)
    out["temperature"] = wx["temperature"] + rng.normal(0, 1.0 + 1.5 * g, shape)
    out["wind_speed"] = np.clip(wx["wind_speed"] + rng.normal(0, 0.5 + 0.7 * g, shape), 0, None)
    out["pressure"] = wx["pressure"] + rng.normal(0, 1.0 + 2.0 * g, shape)
    out["humidity"] = np.clip(wx["humidity"] + rng.normal(0, 4 + 6 * g, shape), 0, 100)
    out["air_density"] = wx["air_density"] + rng.normal(0, 0.005 + 0.007 * g, shape)
    return out

In [ ]:
LEVEL_COLS = ["same1", "same3", "same7", "same7med", "same7max", "lvl24", "lvl72", "lvl336", "peak168", "night168", "last"]


def same_hour(orig, days):
    # y at the same hour of day, 1, 2, ... `days` days back from the target hour, using only data up to the origin.
    m0 = np.ceil(K / 24).astype(int)
    pos = np.asarray(orig)[:, None, None] + K[None, :, None] - 24 * (m0[None, :, None] + np.arange(days)[None, None, :])
    return y[pos]


def make_features(orig, wx):
    orig = np.asarray(orig); no = len(orig)
    f = {"k": np.tile(K, no)}
    f["hour"] = ((hour[orig][:, None] + K[None, :]) % 24).ravel()
    tstamp = ds.to_numpy()[orig][:, None] + K[None, :].astype("timedelta64[h]")
    doy = pd.DatetimeIndex(tstamp.ravel()).dayofyear.to_numpy()
    f["doy_sin"], f["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    for c in WX:
        f[c] = wx[c].reshape(-1)
    s1, s3, s7 = same_hour(orig, 1), same_hour(orig, 3), same_hour(orig, 7)
    f["same1"] = s1[:, :, 0].ravel(); f["same3"] = s3.mean(2).ravel()
    f["same7"] = s7.mean(2).ravel(); f["same7med"] = np.median(s7, 2).ravel(); f["same7max"] = s7.max(2).ravel()
    for w in (24, 72, 168, 336):
        f[f"lvl{w}"] = np.repeat(LEVEL[w][orig], H)
    f["peak168"] = np.repeat(PEAK168[orig], H)
    f["night168"] = np.repeat(NIGHT168[orig], H)
    f["last"] = np.repeat(y[orig], H)
    return pd.DataFrame(f)


def ratio_view(F):
    # Level-free view: every level-like column divided by the trailing 7-day mean.
    G, Lc = F.copy(), F["lvl168"].to_numpy()
    for c in LEVEL_COLS:
        G[c] = F[c] / Lc
    return G.drop(columns=["lvl168"]), Lc


def targets(orig):
    return y[np.asarray(orig)[:, None] + K[None, :]].reshape(-1)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler


class Rule:
    # Same hour, `days` days back; averaged (or the median) when days > 1.
    def __init__(self, name, days, agg="mean"):
        self.name, self.days, self.agg = name, days, agg
    def fit(self, i0): pass
    def predict(self, orig, wx):
        v = same_hour(orig, self.days)
        return v[:, :, 0] if self.days == 1 else (v.mean(2) if self.agg == "mean" else np.median(v, 2))


class Weekly:
    name = "seasonal naive (168 h)"
    def fit(self, i0): pass
    def predict(self, orig, wx):
        return y[np.asarray(orig)[:, None] + K[None, :] - 168]


class HoltWinters:
    def __init__(self, days=21):
        self.name, self.days, self.cache = f"Holt-Winters ({days} d)", days, {}
    def fit(self, i0): pass
    def predict(self, orig, wx):
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        out = []
        for o in orig:
            if o not in self.cache:
                hist = y[o - 24 * self.days + 1: o + 1]
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")            # statsmodels re-enables its own warnings
                        p = ExponentialSmoothing(hist, trend="add", damped_trend=True, seasonal="add",
                                                 seasonal_periods=24).fit().forecast(H)
                except Exception:
                    p = same_hour(np.array([o]), 1)[0, :, 0]
                self.cache[o] = np.clip(p, 0, None)
            out.append(self.cache[o])
        return np.array(out)


class RidgeRatio:
    name = "Ridge on ratio (weather)"
    def _X(self, F):
        G, Lc = ratio_view(F)
        hrs = pd.get_dummies(pd.Categorical(G["hour"].astype(int), categories=range(24)), dtype=float)
        inter = G[WEATHER6].mul(G["daylight"], axis=0).add_suffix("_x_day")
        return np.column_stack([hrs.to_numpy(), G.drop(columns=["hour"]).to_numpy(), inter.to_numpy()]), Lc
    def fit(self, i0):
        orig = ORIG_ALL[ORIG_ALL + H <= i0]
        X, Lc = self._X(make_features(orig, hist_wx(orig)))
        self.sc = StandardScaler().fit(X)
        self.m = Ridge(alpha=10.0).fit(self.sc.transform(X), targets(orig) / Lc)
    def predict(self, orig, wx):
        X, Lc = self._X(make_features(orig, wx))
        return np.clip(self.m.predict(self.sc.transform(X)) * Lc, 0, None).reshape(len(orig), H)


try:
    import xgboost as xgb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
    import xgboost as xgb

LGB_PARAMS = dict(n_estimators=250, learning_rate=0.05, num_leaves=31, min_child_samples=40, subsample=0.8,
                  subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=2, verbose=-1)
XGB_PARAMS = dict(n_estimators=250, learning_rate=0.05, max_depth=6, min_child_weight=5, subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist", n_jobs=2, verbosity=0)


class LGB:
    # One direct model for all 48 horizons. target: 'raw' | 'ratio' (y / trailing 7-day mean) | 'log' (log1p).
    def __init__(self, name, target="ratio", weather=True, half_life_days=None, params=None, objective="regression",
                 alpha=None, engine="lgb", train_days=None, drop=()):
        self.name, self.target, self.weather, self.hl = name, target, weather, half_life_days
        self.drop = list(drop)                                    # extra columns to leave out, e.g. ["cloud_cover"]
        self.engine, self.train_days = engine, train_days          # engine: 'lgb' | 'xgb'; train_days: rolling training window
        self.params = {**(LGB_PARAMS if engine == "lgb" else XGB_PARAMS), **(params or {})}
        self.objective, self.alpha = objective, alpha
    def _view(self, F):
        if self.target == "ratio":
            G, Lc = ratio_view(F)
        else:
            G, Lc = F, F["lvl168"].to_numpy()
        if not self.weather:
            G = G.drop(columns=WEATHER6)
        if self.drop:
            G = G.drop(columns=self.drop)
        return G, Lc
    def fit(self, i0):
        orig = ORIG_ALL[ORIG_ALL + H <= i0]
        if self.train_days:
            orig = orig[orig >= i0 - 24 * self.train_days]
        G, Lc = self._view(make_features(orig, hist_wx(orig)))
        yt = targets(orig)
        t = {"ratio": yt / Lc, "log": np.log1p(yt), "raw": yt}[self.target]
        w = None
        if self.hl:
            w = np.repeat(0.5 ** ((i0 - orig) / (24.0 * self.hl)), H)
        if self.engine == "lgb":
            kw = dict(objective=self.objective, **({"alpha": self.alpha} if self.alpha else {}))
            self.m = lgb.LGBMRegressor(**self.params, **kw).fit(G, t, sample_weight=w)
        else:
            self.m = xgb.XGBRegressor(**self.params).fit(G, t, sample_weight=w)
        self.cols = list(G.columns)
    def predict(self, orig, wx):
        G, Lc = self._view(make_features(orig, wx))
        p = self.m.predict(G[self.cols])
        p = {"ratio": p * Lc, "log": np.expm1(p), "raw": p}[self.target]
        return np.clip(p, 0, None).reshape(len(orig), H)

## 1. Features, validation set and helpers

In [ ]:
if QUICK:
    LGB_PARAMS["n_estimators"] = 60
    XGB_PARAMS["n_estimators"] = 60
BLOCK = 10

# ---- day-level weather features ------------------------------------------------------------------------------
DAILY_COLS = ["d_hum", "d_temp", "d_trange", "d_wind", "d_windmax", "d_pres", "d_hum_chg", "d_temp_chg", "d_wind_chg"]
HUM24 = pd.Series(W["humidity"]).rolling(24).mean().to_numpy()
TMP24 = pd.Series(W["temperature"]).rolling(24).mean().to_numpy()
WND24 = pd.Series(W["wind_speed"]).rolling(24).mean().to_numpy()
if "_make_features_base" not in globals():     # keep the ORIGINAL function once, so re-running this cell cannot make it call itself
    _make_features_base = make_features


def make_features(orig, wx):
    F = _make_features_base(orig, wx)
    orig = np.asarray(orig); no = len(orig)

    def agg(c, fn):
        v = np.empty((no, H))
        for d in (0, 1):
            sl = slice(24 * d, 24 * d + 24)
            v[:, sl] = fn(wx[c][:, sl], axis=1)[:, None]
        return v

    ex = {"d_hum": agg("humidity", np.mean), "d_temp": agg("temperature", np.mean), "d_trange": agg("temperature", np.ptp),
          "d_wind": agg("wind_speed", np.mean), "d_windmax": agg("wind_speed", np.max), "d_pres": agg("pressure", np.mean)}
    ex["d_hum_chg"] = ex["d_hum"] - HUM24[orig][:, None]
    ex["d_temp_chg"] = ex["d_temp"] - TMP24[orig][:, None]
    ex["d_wind_chg"] = ex["d_wind"] - WND24[orig][:, None]
    for k, v in ex.items():
        F[k] = v.reshape(-1)
    return F


# ---- two validation segments: the recent origins (A) and the same season last year (B) -------------------------
segA = ORIG_ALL[-NSEG:]
iB = np.searchsorted(ds.iloc[ORIG_ALL].to_numpy(), np.datetime64("2025-09-26", "ns"))
segB = ORIG_ALL[iB: iB + NSEG]
VAL = np.concatenate([segB, segA]); NVAL = len(VAL)
SEGB, SEGA = np.arange(NSEG), np.arange(NSEG, 2 * NSEG)
actual = y[VAL[:, None] + K[None, :]]
bench = y[VAL[:, None] + K[None, :] - 24 * np.ceil(K / 24).astype(int)[None, :]]      # daily seasonal naive
tpos = VAL[:, None] + K[None, :]
print(f"segment B: {ds.iloc[segB[0]].date()} to {ds.iloc[segB[-1]].date()} | segment A: {ds.iloc[segA[0]].date()} to {ds.iloc[segA[-1]].date()}")


def perturb2(wx, rng, scale=1.0):
    # Simulated weather-forecast error, growing with the horizon (the sizes are an assumption).
    g = (K / H)[None, :]
    out = {c: v.copy() for c, v in wx.items()}
    shape = wx["cloud_cover"].shape
    out["cloud_cover"] = np.clip(np.round((wx["cloud_cover"] + scale * rng.normal(0, 0.15 + 0.25 * g, shape)) * 4) / 4, 0, 1)
    out["temperature"] = wx["temperature"] + scale * rng.normal(0, 1.0 + 1.5 * g, shape)
    out["wind_speed"] = np.clip(wx["wind_speed"] + scale * rng.normal(0, 0.5 + 0.7 * g, shape), 0, None)
    out["pressure"] = wx["pressure"] + scale * rng.normal(0, 1.0 + 2.0 * g, shape)
    out["humidity"] = np.clip(wx["humidity"] + scale * rng.normal(0, 4 + 6 * g, shape), 0, 100)
    out["air_density"] = wx["air_density"] + scale * rng.normal(0, 0.005 + 0.007 * g, shape)
    return out


def rmse_o(P):
    return np.sqrt(((actual - P) ** 2).mean(1))


rb = rmse_o(bench)
half_ = np.arange(NSEG // 2, NSEG) + NSEG


def summarize(P):
    rows = {}
    for nm, p in P.items():
        r = rmse_o(p)
        rows[nm] = {"skill (all)": r.mean() / rb.mean(), "skill B (last autumn)": r[SEGB].mean() / rb[SEGB].mean(),
                    "skill A (recent)": r[SEGA].mean() / rb[SEGA].mean(), "skill last 15": r[-15:].mean() / rb[-15:].mean(),
                    "beats % (all)": 100 * (r < rb).mean(), "last 48h skill": r[-1] / rb[-1]}
    return pd.DataFrame(rows).T.sort_values("skill (all)")

## 2. Gradient-boosting ensemble (rolling backtest)

In [ ]:
NOCLOUD = ["cloud_cover"]
def gbm_models():
    return [LGB("LGB raw", "raw", drop=NOCLOUD), LGB("LGB log1p", "log", drop=NOCLOUD),
            LGB("LGB ratio, recency-weighted", "ratio", half_life_days=365, drop=NOCLOUD),
            LGB("XGB raw", "raw", engine="xgb", drop=NOCLOUD)]

gm = gbm_models()
PG = {m.name: np.zeros((NVAL, H)) for m in gm}
rng = np.random.default_rng(0)
t0 = time.time()
for b in range(0, NVAL, BLOCK):
    idx = VAL[b:b + BLOCK]
    wx = perturb2(hist_wx(idx), rng)
    for m in gm:
        m.fit(idx[0])                                    # only data known at the first origin of the block
        PG[m.name][b:b + BLOCK] = m.predict(idx, wx)
print(f"GBM backtest done in {time.time() - t0:.0f}s")
EG = np.mean(list(PG.values()), axis=0)                  # GBM ensemble, backtest forecasts for the 120 origins
display(summarize({**PG, "GBM ensemble": EG}).round(3))

## 3. Chronos-2 with the weather (rolling backtest)

Temperature, wind speed, pressure, humidity and daylight, past and future. The same simulated weather as for the GBMs. Besides the median, the 2.5%, 10%, 90% and 97.5% quantiles are kept
(used as one of the interval recipes). If Chronos-2 cannot be installed or run, the notebook falls back to the GBM ensemble alone.

In [ ]:
CHM, CH_MODELS, Q, MID = None, [], None, 2
QLEV = [0.025, 0.1, 0.5, 0.9, 0.975]
C5 = ["temperature", "wind_speed", "pressure", "humidity", "daylight"]


class ChronosCov:
    def __init__(self, pipe, name, ctx, covs):
        self.pipe, self.name, self.ctx, self.covs = pipe, name, ctx, covs
    def _inp(self, o, wxj):
        sl_ = slice(o - self.ctx + 1, o + 1)
        return {"target": np.asarray(y[sl_], dtype=np.float32),
                "past_covariates": {c: np.asarray(W[c][sl_], dtype=np.float32) for c in self.covs},
                "future_covariates": {c: np.asarray(wxj[c], dtype=np.float32) for c in self.covs}}
    def quantiles(self, orig, wx):                       # (n, H, 5): 2.5%, 10%, 50%, 90%, 97.5%
        out = []
        for j, o in enumerate(orig):
            q, _ = self.pipe.predict_quantiles([self._inp(o, {c: wx[c][j] for c in self.covs})], prediction_length=H, quantile_levels=QLEV)
            out.append(np.clip(np.asarray(q[0]).reshape(-1, H, len(QLEV))[0], 0, None))
        return np.array(out)


if RUN_FOUNDATION:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting==2.3.2"], capture_output=True, text=True)
        import torch
        from chronos import BaseChronosPipeline
        pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda" if torch.cuda.is_available() else "cpu")
        CH_MODELS = [ChronosCov(pipe, "Chronos-2 + weather, ctx 2048", 2048, C5), ChronosCov(pipe, "Chronos-2 + weather, ctx 1024", 1024, C5)]
        try:
            test = CH_MODELS[0].quantiles(np.array([VAL[0]]), hist_wx(np.array([VAL[0]])))
            assert test.shape == (1, H, len(QLEV)) and np.isfinite(test).all()
        except Exception as e1:                                  # the 2.5% / 97.5% levels are not available: fall back to the proven 10/50/90
            print("5 quantile levels failed (", type(e1).__name__, "), using 10/50/90 only; the Chronos-2 quantile interval recipe is off")
            QLEV = [0.1, 0.5, 0.9]
            test = CH_MODELS[0].quantiles(np.array([VAL[0]]), hist_wx(np.array([VAL[0]])))
            assert test.shape == (1, H, len(QLEV)) and np.isfinite(test).all()
        MID = QLEV.index(0.5)
        QALL = {m.name: np.zeros((NVAL, H, len(QLEV))) for m in CH_MODELS}
        rng_c = np.random.default_rng(0)
        t0 = time.time()
        for b in range(0, NVAL, BLOCK):
            idx = VAL[b:b + BLOCK]
            wx = perturb2(hist_wx(idx), rng_c)
            for m in CH_MODELS:
                QALL[m.name][b:b + BLOCK] = m.quantiles(idx, wx)
        print(f"Chronos-2 backtest done in {time.time() - t0:.0f}s")
        PCH = {nm: q[..., MID] for nm, q in QALL.items()}
        CHM = np.mean(list(PCH.values()), axis=0)
        Q = QALL["Chronos-2 + weather, ctx 2048"] if len(QLEV) == 5 else None
        display(summarize({**PCH, "Chronos-2 average": CHM}).round(3))
    except Exception as e:
        CHM, CH_MODELS, Q = None, [], None
        print("Chronos-2 skipped, GBM ensemble only:", type(e).__name__, str(e)[:300])
else:
    print("RUN_FOUNDATION is off: GBM ensemble only")

## 4. The blend

In [ ]:
WGe = WG if CHM is not None else 1.0
Pbl_raw = WGe * EG + (1 - WGe) * (CHM if CHM is not None else EG)
dl_all = W["daylight"][tpos]                              # 1 = day, 0 = night, for every backtest hour
Pbl = Pbl_raw * np.where(dl_all == 0, NF, DF)             # always rebuilt from Pbl_raw: re-running never scales twice
rows = {"GBM ensemble": EG, f"BLEND ({WGe:.2f} GBM + {1 - WGe:.2f} Chronos-2)": Pbl}
if CHM is not None:
    rows["Chronos-2 average"] = CHM
    for w in (0.3, 0.7):
        rows[f"blend, {w} GBM"] = w * EG + (1 - w) * CHM
display(summarize(rows).round(3))
mape_ = lambda a, f: np.mean(np.abs((a - f) / a)) * 100
a2 = actual[:, 24:]
print(f"hours 25-48, RMSE / MAPE %: before night/day scaling {np.sqrt(((a2 - Pbl_raw[:, 24:]) ** 2).mean()):.1f} / {mape_(a2, Pbl_raw[:, 24:]):.1f}"
      f" | after (NF={NF}, DF={DF}) {np.sqrt(((a2 - Pbl[:, 24:]) ** 2).mean()):.1f} / {mape_(a2, Pbl[:, 24:]):.1f}")
print("blend vs benchmark per origin: beats it on", round(100 * (rmse_o(Pbl) < rb).mean()), "% of", NVAL, "origins")

## 5. The interval

Evaluated on the recent origins (segment A), with online calibration (origin `j` uses the up to 60 earlier origins whose errors are fully known, `j-2` and before).
The score is the mean of `log(Winkler of the band on hours 25-48 / Winkler of the benchmark's own band)`, which is scale-free, so volatile high-output windows do not dominate.

In [ ]:
C0 = 50.0
def group_ids(daylight, kind="12"):
    dl = np.asarray(daylight).astype(int)
    day2 = np.broadcast_to((K - 1) // 24, dl.shape)
    if kind == "4":
        return dl * 2 + day2
    return np.broadcast_to(((K - 1) % 24) // 4, dl.shape) + 6 * day2


def band_from(pc, ac, dlc, pn, dln, kind="12", s=1.0, mode="abs", q=0.95):
    Gc, Gn = group_ids(dlc, kind), group_ids(dln, kind)
    ae = np.abs(ac - pc)
    rel = ae / (C0 + pc)
    def qk(v):
        pooled = np.quantile(v, q)
        return np.array([np.quantile(v[Gc == k], q) if (Gc == k).sum() >= 15 else pooled for k in range(Gc.max() + 1)])
    ha, hr = qk(ae)[Gn], qk(rel)[Gn] * (C0 + pn)
    half = s * {"abs": ha, "rel": hr, "mix": 0.5 * (ha + hr)}[mode]
    return np.clip(pn - half, 0, None), pn + half


pBl, aA, dlA = Pbl[SEGA], actual[SEGA], W["daylight"][tpos][SEGA]
J0 = 22 if NSEG >= 40 else NSEG // 2
idx = np.arange(J0, NSEG)
if Q is not None:
    medA = Q[SEGA][..., 2]
    dnA, upA = np.clip(medA - Q[SEGA][..., 0], 0, None), np.clip(Q[SEGA][..., 4] - medA, 0, None)


def band_online(recipe, back=60):
    mode, kind, s = recipe
    if mode == "cq":                                       # Chronos-2 quantile widths around the blended forecast
        return np.clip(pBl - s * dnA, 0, None), pBl + s * upA
    lo, hi = np.full((NSEG, H), np.nan), np.full((NSEG, H), np.nan)
    for j in range(J0, NSEG):
        past = np.arange(max(0, j - back), j - 1)
        lo[j], hi[j] = band_from(pBl[past], aA[past], dlA[past], pBl[j], dlA[j], kind, s, mode)
    return lo, hi


sn = [snaive(y[:o + 1]) for o in segA]
bn_lo, bn_hi = np.array([a[1] for a in sn]), np.array([a[2] for a in sn])
w2 = lambda lo, hi: np.array([winkler(aA[j, 24:], lo[j, 24:], hi[j, 24:]) for j in idx])
wb = w2(bn_lo, bn_hi)
recipes = [(m, k, s) for m in ("abs", "rel", "mix") for k in ("4", "12") for s in (0.9, 1.0, 1.1, 1.2, 1.3, 1.4)]
if Q is not None:
    recipes += [("cq", "-", s) for s in (0.8, 1.0, 1.2, 1.4, 1.6)]
rows = {}
for rc in recipes:
    lo, hi = band_online(rc)
    w = w2(lo, hi)
    inside = (aA[idx, 24:] >= lo[idx, 24:]) & (aA[idx, 24:] <= hi[idx, 24:])
    rows[rc] = {"mean log(W / benchmark band)": np.mean(np.log(w / wb)), "median ratio": np.median(w / wb),
                "mean Winkler day 2": w.mean(), "coverage % day 2": 100 * inside.mean(),
                "coverage % all 48 h": 100 * ((aA[idx] >= lo[idx]) & (aA[idx] <= hi[idx])).mean(), "mean width": (hi[idx] - lo[idx]).mean()}
tabI = pd.DataFrame(rows).T
tabI.index = [f"{m}, {k}, s={s}" for m, k, s in tabI.index]
ok = tabI[tabI["coverage % day 2"] >= 94].sort_values("mean log(W / benchmark band)")
display(ok.head(12).round(3))
best = ok.index[0]
rc_best = [rc for rc in recipes if f"{rc[0]}, {rc[1]}, s={rc[2]}" == best][0]
if FORCE_RECIPE is not None:
    rc_best = FORCE_RECIPE
    best = f"{rc_best[0]}, {rc_best[1]}, s={rc_best[2]}"
    print("recipe forced by FORCE_RECIPE:", best)
    display(tabI.loc[[best]].round(3))
print("chosen recipe:", best, "| benchmark band for reference: mean Winkler day 2", round(wb.mean()))
print("the choice is made on the same origins it is scored on: a little optimistic. Look at the top rows: if they are close, any of them is fine.")

## 6. Final fit on all history, forecast, interval and the file

In [ ]:
wx_final = {c: wfc[c].to_numpy(float)[None, :] for c in WX}
O_FINAL = np.array([n - 1])                              # the last row of train.csv is 00:00: the origin of the forecast
gm_final = gbm_models()
fc_g = np.mean([(m.fit(n - 1), m.predict(O_FINAL, wx_final)[0])[1] for m in gm_final], axis=0)
if CHM is not None:
    Qf = [m.quantiles(O_FINAL, wx_final)[0] for m in CH_MODELS]                      # each (H, 5)
    fc_c = np.mean([q[:, MID] for q in Qf], axis=0)
    fc = WGe * fc_g + (1 - WGe) * fc_c
    qf = Qf[0]
else:
    fc = fc_g
fc = np.clip(fc, 0, None)
fc = fc * np.where(wfc["daylight"].to_numpy() == 0, NF, DF)   # same night/day factors as in the backtest

mode, kind, s = rc_best
if mode == "cq":
    lo = np.clip(fc - s * np.clip(qf[:, 2] - qf[:, 0], 0, None), 0, None); hi = fc + s * np.clip(qf[:, 4] - qf[:, 2], 0, None)
else:
    lo, hi = band_from(pBl, aA, dlA, fc, wfc["daylight"].to_numpy(), kind, s, mode)

sub = pd.DataFrame({"ds": wfc["ds"].dt.strftime("%Y-%m-%d %H:%M:%S"), "forecast": fc, "lower_95": lo, "upper_95": hi})
assert len(sub) == 48 and list(sub.columns) == ["ds", "forecast", "lower_95", "upper_95"] and sub.notna().all().all()
assert (sub.lower_95 >= 0).all() and (sub.lower_95 <= sub.forecast).all() and (sub.forecast <= sub.upper_95).all()
assert sub["ds"].iloc[0] == "2026-10-27 01:00:00" and sub["ds"].iloc[-1] == "2026-10-29 00:00:00"
print("checks passed | recipe:", best, "| mean width", round((sub.upper_95 - sub.lower_95).mean()))
print(f"day totals: GBM ensemble {fc_g[:24].sum():.0f} / {fc_g[24:].sum():.0f}", end="")
if CHM is not None:
    print(f" | Chronos-2 {fc_c[:24].sum():.0f} / {fc_c[24:].sum():.0f}", end="")
print(f" | final (after NF/DF) {fc[:24].sum():.0f} / {fc[24:].sum():.0f} | yesterday {y[n - 24:n].sum():.0f}")
display(sub.round(1).head(6))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(ds.iloc[-24 * 5:], y[-24 * 5:], color="black", label="last 5 days")
ax.plot(wfc["ds"], fc_g, color="C1", lw=1, label="GBM ensemble")
if CHM is not None:
    ax.plot(wfc["ds"], fc_c, color="C2", lw=1, label="Chronos-2 + weather")
ax.plot(wfc["ds"], sub.forecast, color="C0", lw=2, label="final forecast")
ax.fill_between(wfc["ds"], sub.lower_95, sub.upper_95, color="C0", alpha=0.2, label="95% interval")
ax.legend(fontsize=8); ax.set_title("final forecast"); plt.show()

if SAVE:
    fn = f"{SURNAME}_project.csv"
    sub.to_csv(fn, index=False)
    print("saved", fn)
    try:
        from google.colab import files
        files.download(fn)
    except ImportError:
        pass

## 7. Rolling-window plot of the submission

In [ ]:
lo_o, hi_o = band_online(rc_best)
Jo = idx
fig, axes = plt.subplots(3, 1, figsize=(14, 13))
tt = np.concatenate([ds.iloc[segA[j] + K[24:]].to_numpy() for j in Jo])
act = np.concatenate([aA[j, 24:] for j in Jo]); fcs = np.concatenate([pBl[j, 24:] for j in Jo])
los = np.concatenate([lo_o[j, 24:] for j in Jo]); his = np.concatenate([hi_o[j, 24:] for j in Jo])
bn_ = np.concatenate([bench[SEGA][j, 24:] for j in Jo])
ax = axes[0]
ax.fill_between(tt, los, his, color="C0", alpha=0.2, label="95% band (online)")
ax.plot(tt, act, color="black", lw=1.2, label="actual"); ax.plot(tt, fcs, color="C0", lw=1.2, label="blend (hours 25-48 of each origin)")
ax.plot(tt, bn_, color="gray", lw=0.8, ls="--", label="benchmark")
miss = (act < los) | (act > his)
ax.scatter(tt[miss], act[miss], color="red", s=14, zorder=5, label=f"outside the band ({100 * miss.mean():.1f}% of hours)")
ax.legend(fontsize=8, ncol=2); ax.set_title("rolling-origin forecasts, hours 25-48 of each midnight origin (what the leaderboard scores)")

rr = rmse_o(Pbl) / rb
ax = axes[1]
ax.bar(np.arange(NVAL), rr, color=np.where(rr < 1, "tab:green", "tab:red"), alpha=0.8)
ax.plot(np.arange(NVAL), pd.Series(rr).rolling(10, min_periods=5).mean(), color="black", lw=2, label="rolling mean of 10 origins")
ax.axhline(1, color="k", ls="--"); ax.axvline(NSEG - 0.5, color="gray"); ax.set_ylim(0, min(4, rr.max() * 1.05))
ax.set_title(f"RMSE of the blend / RMSE of the benchmark, per origin: beats the benchmark on {100 * (rr < 1).mean():.0f}% of {NVAL} origins (left: last autumn, right: recent)")
ax.legend(fontsize=8)

wa = w2(lo_o, hi_o)
cov2 = np.array([100 * ((aA[j, 24:] >= lo_o[j, 24:]) & (aA[j, 24:] <= hi_o[j, 24:])).mean() for j in Jo])
ax = axes[2]
ax.bar(Jo - Jo[0], wa, color="tab:blue", alpha=0.6, label="Winkler, hours 25-48")
ax.bar(Jo - Jo[0], wb, color="gray", alpha=0.35, width=0.4, label="benchmark band's Winkler")
ax2 = ax.twinx()
ax2.plot(Jo - Jo[0], pd.Series(cov2).rolling(10, min_periods=5).mean(), color="tab:orange", lw=2, label="coverage, rolling 10 origins (%)")
ax2.axhline(95, color="tab:orange", ls=":"); ax2.set_ylim(50, 101)
ax.set_title(f"day-2 Winkler per recent origin ({best}); overall day-2 coverage {cov2.mean():.1f}%")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, fontsize=8)
plt.tight_layout(); plt.savefig("rolling_window_submission.png", dpi=150); plt.show()